# 자연어 처리 2일차 (1) — 한글 형태소 단위 토큰화 : Okt · Mecab (morphs · pos · nouns)

수업 필기 `단어사전02` 를 **실행 가능하도록 정리 · 개선한 노트**입니다.
(교재 `자연어처리` 교재9 p.32~40 — KoNLPy · Okt 형태소 분석기 · 품사 정리 · mecab 형태소 분석기 · 품사 정리 · 실습)

- 헌법 텍스트로 명사 빈도를 세고 워드클라우드를 그리는 실습(필기 `문서_마이닝`)은 [`constitution_word_frequency.ipynb`](./constitution_word_frequency.ipynb) 에 이어서 정리했습니다.
- 어제 배운 WordPiece 단어 사전(필기 `단어사전01`)은 [Day 039 (11-03)](../../2026-11-03/lecture/wordpiece_vocab_nsmc.ipynb) 에 정리되어 있습니다.
- 원본에서 고친 부분은 **📝 고친 부분** 으로 따로 적어 두었습니다. **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 있습니다.

| 기능 | Okt (`konlpy`) | Mecab (`python-mecab-ko`) | 결과 |
|---|---|---|---|
| 형태소 분석 | `okt.morphs(text)` | `mecab.morphs(text)` | `[형태소1, 형태소2, ...]` |
| 형태소 + 품사 | `okt.pos(text)` | `mecab.pos(text)` | `[(형태소, 품사), ...]` |
| 명사 추출 | `okt.nouns(text)` | `mecab.nouns(text)` | `[명사1, 명사2, ...]` |

## 0. 설치 · 실행 방법

| 패키지 | 이 노트에서 쓰는 것 | 설치 |
|---|---|---|
| `konlpy` | **Okt** 형태소 분석기 · 헌법 말뭉치 `kolaw` | `pip install konlpy` — **Java(JDK)** 가 있어야 한다. Colab 은 기본 포함, Windows · Mac 은 JDK 설치 후 `JAVA_HOME` 설정 |
| `python-mecab-ko` | **Mecab** 형태소 분석기 (한국어 사전 포함) | `pip install python-mecab-ko` — 미리 만든 설치 파일이 없는 환경(예 : Python 3.13)에서는 직접 빌드하므로 리눅스 · Colab 에서 `apt-get install mecab libmecab-dev` 를 먼저 (필기와 같은 방법) |
| `kiwipiepy` | **Kiwi** 형태소 분석기 (비교용) | `pip install kiwipiepy` |
| `nltk` | `word_tokenize` · `RegexpTokenizer` | `pip install nltk` → 노트 안에서 `nltk.download('punkt_tab')` |
| `wordcloud` | 워드클라우드 | `pip install wordcloud` + **한글 폰트** (Colab : `apt-get install fonts-nanum`) |

**실행 순서** : ① 아래 설치 셀의 `#` 를 지우고 한 번 실행 (이미 설치돼 있으면 건너뜀) → ② 맨 위부터 **[런타임] → [모두 실행]** (셀을 건너뛰거나 거꾸로 실행하면 변수가 바뀐 채 남는다). 헌법 텍스트는 KoNLPy 에 들어 있어서 **따로 받을 파일이 없다**.

In [1]:
# ① 설치 (Colab · 처음 한 번)
# !apt-get install -y -qq mecab libmecab-dev fonts-nanum   # Mecab 빌드 도구 · 한글 폰트
# !pip install -q konlpy python-mecab-ko kiwipiepy nltk wordcloud

In [2]:
# 그래프 한글 폰트 설정 (운영체제별)
import platform
import matplotlib.pyplot as plt

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

In [3]:
import warnings
warnings.filterwarnings('ignore')

from konlpy.tag import Okt
from mecab import MeCab                      # 교재 : from mecab import MeCab as Mecab
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt_tab', quiet=True)

okt = Okt()                                  # 형태소 분석기 객체 생성
mecab = MeCab()

> **📝 고친 부분**
> - 객체 이름 `mc` · `mecap` · 제목 `MeCap` → 도구 이름은 **MeCab** (Me + Cab) — 이 노트는 `mecab` 으로 통일
> - `!pip install konlpy` 와 `!apt-get ...` · `!pip install python-mecab-ko` 를 한 셀에 몰아 두었다 → 설치는 맨 위에 한 번, 실행 환경에 이미 있으면 주석으로 둔다

## 1. 같은 문장 · 네 가지 토큰화 (교재 p.32)

In [4]:
text = '한글 형태소 분석기로 테스트를 하였다, 작동은 정상이다.'

print('공백 split()  :', text.split())             # 띄어쓰기 — 조사 · 문장부호가 붙은 채
print('word_tokenize :', word_tokenize(text))      # 띄어쓰기 + 문장부호만 분리
print('Okt.morphs    :', okt.morphs(text))         # 형태소
print('Mecab.morphs  :', mecab.morphs(text))

공백 split()  : ['한글', '형태소', '분석기로', '테스트를', '하였다,', '작동은', '정상이다.']
word_tokenize : ['한글', '형태소', '분석기로', '테스트를', '하였다', ',', '작동은', '정상이다', '.']


Okt.morphs    : ['한글', '형태소', '분석', '기', '로', '테스트', '를', '하였다', ',', '작동', '은', '정상', '이다', '.']
Mecab.morphs  : ['한글', '형태소', '분석기', '로', '테스트', '를', '하', '였', '다', ',', '작동', '은', '정상', '이', '다', '.']


- `split()` · `word_tokenize()` 는 `분석기로` · `테스트를` 처럼 **조사가 붙은 어절** 그대로 → 같은 '테스트' 가 조사마다 다른 단어가 된다(Day 039 OOV)
- Okt · Mecab 은 `분석 / 기 / 로`, `테스트 / 를` 처럼 **형태소**로 나눈다. Okt 는 `하였다` 를 통째로, Mecab 은 `하 / 였 / 다` 로 더 잘게 나눈다

![토큰화 비교 애니메이션](./images/anim_01_tokenize_compare.gif)

## 2. Okt — `morphs` · `pos` · `nouns` (교재 p.33~36)

In [5]:
print('morphs :', okt.morphs(text))
print('pos    :', okt.pos(text))
print('nouns  :', okt.nouns(text))

# Okt 만의 옵션 : norm(정규화) · stem(어간 추출 = 기본형으로)
slang = '이 영화 진짜 잼있었닼ㅋㅋ 강추합니당'
print()
print('기본          :', okt.morphs(slang))
print('norm=True     :', okt.morphs(slang, norm=True))
print('norm + stem   :', okt.pos(slang, norm=True, stem=True))

morphs : ['한글', '형태소', '분석', '기', '로', '테스트', '를', '하였다', ',', '작동', '은', '정상', '이다', '.']
pos    : [('한글', 'Noun'), ('형태소', 'Noun'), ('분석', 'Noun'), ('기', 'Noun'), ('로', 'Josa'), ('테스트', 'Noun'), ('를', 'Josa'), ('하였다', 'Verb'), (',', 'Punctuation'), ('작동', 'Noun'), ('은', 'Josa'), ('정상', 'Noun'), ('이다', 'Josa'), ('.', 'Punctuation')]
nouns  : ['한글', '형태소', '분석', '기', '테스트', '작동', '정상']

기본          : ['이', '영화', '진짜', '잼있었', '닼', 'ㅋㅋ', '강', '추합니당']
norm=True     : ['이', '영화', '진짜', '잼있었다', 'ㅋㅋ', '강', '추합니다']
norm + stem   : [('이', 'Noun'), ('영화', 'Noun'), ('진짜', 'Noun'), ('재다', 'Verb'), ('ㅋㅋ', 'KoreanParticle'), ('강', 'Noun'), ('추하다', 'Adjective')]


- Okt 의 `stem=True` 는 `잼있었` → `재다`, `강추합니당` → `강` + `추하다` 처럼 **신조어 · 줄임말의 기본형을 엉뚱하게** 만든다 — 결과를 꼭 눈으로 확인

**Okt 품사 (교재 p.35 · `okt.tagset`)**

| 품사 | 뜻 | 품사 | 뜻 |
|---|---|---|---|
| Noun | 명사 · 대명사 · 수사 | Josa | 조사 (를, 은) |
| Verb | 동사 | Eomi | 어미 (요, 어) |
| Adjective | 형용사 | PreEomi | 선어말어미 (었) |
| Determiner | 관형사 | Suffix | 접미사 |
| Adverb | 부사 (매우, 빨리) | Punctuation | 문장부호 |
| Conjunction | 접속사 | KoreanParticle | 자음 · 모음만 (ㅋㅋ) |
| Exclamation | 감탄사 | Foreign · Alpha · Number | 외국어 · 알파벳 · 숫자 |

교재 표에는 트위터용 `Hashtag` · `ScreenName` · `Email` · `URL` 도 있다. 그런데 실제 결과에는 교재 표에도 `okt.tagset` 에도 없는 **`Modifier`**(관형어, 예 : `오는`)가 나오고, `okt.tagset` 에는 **`Unknown`**(미등록어)도 있다.

> **📝 고친 부분** (교재 p.35) — Determiner(관형사) 예시에 들어 있는 `잘` 은 문법적으로 **부사**다(관형사 예 : 새 · 헌 · 첫 · 이 · 그 · 저). 게다가 아래처럼 실제 Okt 는 `새 · 첫 · 그` 를 **Noun**, `잘` 을 **Verb** 로 붙인다 — 표의 품사가 그대로 나온다고 믿지 말 것

In [6]:
print(okt.tagset)
for w in ['잘 했다', '새 책', '첫 수업', '그 사람']:
    print(f'{w:6s} →', okt.pos(w))

{'Adjective': '형용사', 'Adverb': '부사', 'Alpha': '알파벳', 'Conjunction': '접속사', 'Determiner': '관형사', 'Eomi': '어미', 'Exclamation': '감탄사', 'Foreign': '외국어, 한자 및 기타기호', 'Hashtag': '트위터 해쉬태그', 'Josa': '조사', 'KoreanParticle': '(ex: ㅋㅋ)', 'Noun': '명사', 'Number': '숫자', 'PreEomi': '선어말어미', 'Punctuation': '구두점', 'ScreenName': '트위터 아이디', 'Suffix': '접미사', 'Unknown': '미등록어', 'Verb': '동사'}
잘 했다   → [('잘', 'Verb'), ('했다', 'Verb')]
새 책    → [('새', 'Noun'), ('책', 'Noun')]
첫 수업   → [('첫', 'Noun'), ('수업', 'Noun')]
그 사람   → [('그', 'Noun'), ('사람', 'Noun')]


## 3. Mecab — `morphs` · `pos` · `nouns` (교재 p.37~40)

Mecab 의 품사는 **세종 품사 태그**(교재 p.39)를 따른다. 형태소가 붙어 한 덩어리로 나오면 `XSV+EF` 처럼 `+` 로 이어 쓴다.

In [7]:
print('morphs :', mecab.morphs(text))
print('pos    :', mecab.pos(text))
print('nouns  :', mecab.nouns(text))

morphs : ['한글', '형태소', '분석기', '로', '테스트', '를', '하', '였', '다', ',', '작동', '은', '정상', '이', '다', '.']
pos    : [('한글', 'NNG'), ('형태소', 'NNG'), ('분석기', 'NNG'), ('로', 'JKB'), ('테스트', 'NNG'), ('를', 'JKO'), ('하', 'VV'), ('였', 'EP'), ('다', 'EF'), (',', 'SC'), ('작동', 'NNG'), ('은', 'JX'), ('정상', 'NNG'), ('이', 'VCP'), ('다', 'EF'), ('.', 'SF')]
nouns  : ['한글', '형태소', '분석기', '테스트', '작동', '정상']


| 대분류 | 태그 | 뜻 | 대분류 | 태그 | 뜻 |
|---|---|---|---|---|---|
| 체언 | NNG · NNP · NNB | 일반 · 고유 · 의존 명사 | 조사 | JKS · JKO · JKB · JX | 주격 · 목적격 · 부사격 · 보조사 |
| | NR · NP | 수사 · 대명사 | 선어말어미 | EP | 였 · 시 |
| 용언 | VV · VA · VX | 동사 · 형용사 · 보조 용언 | 어말어미 | EF · EC · ETM · ETN | 종결 · 연결 · 관형형 · 명사형 |
| | VCP · VCN | 긍정 · 부정 지정사(이다 · 아니다) | 접사 | XPN · XSN · XSV · XSA | 접두사 · 명사/동사/형용사 파생 접미사 |
| 수식언 | MM · MAG · MAJ | 관형사 · 일반 부사 · 접속 부사 | 부호 | SF · SC · SS | 마침표 · 쉼표 · 괄호 |

![품사 태깅 애니메이션](./images/anim_02_pos_tagging.gif)

## 4. 문장이 틀리면 분석도 틀린다 — 필기 예문

In [8]:
memo = '오는말이 고아야 가는말이 곱게 전달된다.'       # 필기 예문 (띄어쓰기 · 맞춤법 그대로)
fixed = '가는 말이 고와야 오는 말이 곱다.'            # 바른 속담

for s in [memo, fixed]:
    print('문장 :', s)
    print('  Okt   pos  :', okt.pos(s))
    print('  Mecab pos  :', mecab.pos(s))
    print('  Okt   nouns:', okt.nouns(s), '| Mecab nouns:', mecab.nouns(s))
    print()

문장 : 오는말이 고아야 가는말이 곱게 전달된다.
  Okt   pos  : [('오는', 'Modifier'), ('말', 'Noun'), ('이', 'Josa'), ('고아', 'Noun'), ('야', 'Josa'), ('가는말이', 'Verb'), ('곱', 'Noun'), ('게', 'Josa'), ('전달', 'Noun'), ('된다', 'Verb'), ('.', 'Punctuation')]
  Mecab pos  : [('오', 'VV'), ('는', 'ETM'), ('말', 'NNG'), ('이', 'VCP'), ('고', 'EC'), ('아', 'VV'), ('야', 'EC'), ('가', 'VX'), ('는', 'ETM'), ('말', 'NNG'), ('이', 'JKS'), ('곱', 'VA'), ('게', 'EC'), ('전달', 'NNG'), ('된다', 'XSV+EF'), ('.', 'SF')]
  Okt   nouns: ['말', '고아', '곱', '전달'] | Mecab nouns: ['말', '말', '전달']

문장 : 가는 말이 고와야 오는 말이 곱다.


  Okt   pos  : [('가는', 'Verb'), ('말', 'Noun'), ('이', 'Josa'), ('고', 'Noun'), ('와야', 'Verb'), ('오는', 'Verb'), ('말', 'Noun'), ('이', 'Josa'), ('곱다', 'Adjective'), ('.', 'Punctuation')]
  Mecab pos  : [('가', 'VV'), ('는', 'ETM'), ('말', 'NNG'), ('이', 'VCP'), ('고', 'EC'), ('와야', 'VV+EC'), ('오', 'VV'), ('는', 'ETM'), ('말', 'NNG'), ('이', 'JKS'), ('곱', 'VA'), ('다', 'EF'), ('.', 'SF')]
  Okt   nouns: ['말', '고', '말'] | Mecab nouns: ['말', '말']



- 필기 예문에서 Okt 는 `고아`(孤兒) · `곱` 을 **명사**로, `가는말이` 를 동사 하나로 잘못 본다. Mecab 도 `말 / 이(VCP, 지정사)` · `고 / 아 / 야` 로 엉킨다
- 맞춤법을 고친 문장도 **`고와야`(곱 + 아야, ㅂ 불규칙 활용)** 는 둘 다 틀린다 — Okt 는 `고`(명사) + `와야`(동사), Mecab 은 앞의 `말이` 까지 `이(VCP)` · `고(EC)` · `와야(VV+EC)` 로 본다. `곱다` 는 둘 다 맞게 나눈다
- 같은 문장을 Kiwi 는 `곱(VA-I) + 어야` 로 맞힌다(6절) → **불규칙 활용**은 분석기마다 실력이 다르고, 맞춤법 · 띄어쓰기가 틀리면 모두 더 크게 틀린다

> **📝 고친 부분**
> - 필기 예문 `오는말이 고아야 가는말이 곱게` → 속담은 **가는 말이 고와야 오는 말이 곱다** (순서 · `고와야` 맞춤법 · 띄어쓰기)
> - 필기의 `okt.pos(text)` 셀 출력이 바로 위의 `text`(`한글 형태소 분석기로…`)가 아닌 **다른 문장**의 결과 — 셀을 위아래로 오가며 실행한 흔적. 위에서부터 다시 실행하면 출력이 바뀐다 → 제출 전 **[런타임] → 모두 실행** 으로 확인
> - 제목 `'형태소'(MeCap) 추출` 아래 셀은 `okt.nouns` 를 쓰고, `print("형태소:")` 인데 출력은 `형태소+명사:` — 코드와 출력이 맞지 않음
> - 주석 `MeCap은 Okt 보다 빠르고 정확하다` → **빠른 것은 맞다**(5절 측정). 정확도는 문장 · 사전에 따라 다르다 — 위 예문은 둘 다 틀렸다
> - 오타 : `분장으로 부터` → 문장으로부터, `기중으로` → 기준으로, `word_wokenize` → word_tokenize, `taging` → tagging

## 5. 명사 추출 비교 · 속도

In [9]:
import time
from konlpy.corpus import kolaw
from kiwipiepy import Kiwi

law = kolaw.open('constitution.txt').read()        # KoNLPy 에 들어 있는 대한민국 헌법 (18,884자)
kiwi = Kiwi()

analyzers = {
    'Okt': okt.nouns,
    'Mecab': mecab.nouns,
    'Kiwi': lambda s: [t.form for t in kiwi.tokenize(s) if t.tag.startswith('NN')],
}
secs = {}
for name, f in analyzers.items():
    f('워밍업')                                     # 첫 호출(JVM · 모델 로딩) 시간 제외
    t0 = time.perf_counter()
    nouns = f(law)
    secs[name] = time.perf_counter() - t0
    print(f'{name:6s} 명사 {len(nouns):,}개 · 종류 {len(set(nouns)):,} · {secs[name]:.2f}초')
print(f"Okt 는 Mecab 보다 {secs['Okt'] / secs['Mecab']:.1f}배 · Kiwi 보다 {secs['Okt'] / secs['Kiwi']:.1f}배 느림")

Okt    명사 3,882개 · 종류 929 · 1.61초


Mecab  명사 3,609개 · 종류 866 · 0.52초


Kiwi   명사 3,644개 · 종류 856 · 0.40초
Okt 는 Mecab 보다 3.1배 · Kiwi 보다 4.0배 느림


같은 헌법 전문에서 **Mecab · Kiwi 가 Okt 보다 여러 배 빠르다**(위 출력, Okt 는 Java 를 거친다 · 첫 호출은 빼고 잰 값이라 컴퓨터마다 조금씩 다르다). 대량 텍스트(리뷰 수십만 건)에는 Mecab · Kiwi 가 현실적이다.

## 6. 기타 — 정규식 토크나이저 · Kiwi (교재 p.32 '기타')

In [10]:
from nltk.tokenize import RegexpTokenizer

sentence = "sorry, I can't go there"
print(r'[\w]+     :', RegexpTokenizer(r'[\w]+').tokenize(sentence))       # 글자 · 숫자만 → can't 가 can / t 로 쪼개짐
print(r"[\w']+    :", RegexpTokenizer(r"[\w']+").tokenize(sentence))      # 작은따옴표 포함 → can't 유지
print('word_tokenize:', word_tokenize(sentence))                          # ca / n't (축약을 문법 단위로)
print()
print('Kiwi :', [(t.form, t.tag) for t in kiwi.tokenize(fixed)])

[\w]+     : ['sorry', 'I', 'can', 't', 'go', 'there']
[\w']+    : ['sorry', 'I', "can't", 'go', 'there']
word_tokenize: ['sorry', ',', 'I', 'ca', "n't", 'go', 'there']

Kiwi : [('가', 'VV'), ('는', 'ETM'), ('말', 'NNG'), ('이', 'JKS'), ('곱', 'VA-I'), ('어야', 'EC'), ('오', 'VV'), ('는', 'ETM'), ('말', 'NNG'), ('이', 'JKS'), ('곱', 'VA-I'), ('다', 'EF'), ('.', 'SF')]


- `RegexpTokenizer(r"[\w]+")` 는 '단어 글자가 1개 이상 이어진 것' 만 토큰으로 → 아포스트로피에서 `can` / `t` 로 끊긴다. 패턴에 `'` 를 넣으면 `can't` 유지
- Kiwi 는 별도 설치 없이 `pip` 만으로 쓰는 한국어 분석기(Day 039 에서 사용) — 세종 계열 태그

## 📝 고친 부분 정리 (필기 `단어사전02`)

| 원본 | 고친 것 |
|---|---|
| `MeCap` · `mc` · `mecap` | **MeCab** · `mecab` |
| 예문 `오는말이 고아야 가는말이 곱게` | 가는 말이 고와야 오는 말이 곱다 (둘 다 비교) |
| `okt.pos(text)` 출력이 다른 문장 결과 | 위에서부터 다시 실행 |
| '(MeCap) 추출' 제목 아래 `okt.nouns`, 출력 문구 불일치 | 제목 · 코드 · 출력 맞춤 |
| 'MeCab 이 더 정확' | 빠른 것은 맞음(5절 측정) · 정확도는 문장마다 (예문은 둘 다 틀림) |
| `분장으로 부터` · `기중으로` · `word_wokenize` · `taging` | 문장으로부터 · 기준으로 · word_tokenize · tagging |
| (교재 p.35) Determiner 예 `잘` · 표에 없는 `Modifier` | `잘` 은 부사 · 실제 Okt 는 관형사를 Noun 으로 붙이고 `Modifier` · `Unknown` 도 낸다 |